# **Install Dependencies**

In [1]:
!pip install torch transformers sentencepiece language-tool-python spacy rouge-score nltk sentence-transformers
!python -m spacy download en_core_web_sm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 2.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=7a2ec58403d70a1830a3b9abd8b4634fdcaaa5a5376424759cf7605a0e511579
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Step 2: Define the Core Paraphraser Class
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import language_tool_python
import spacy

In [4]:
class AIParaphraser:
    """
    AI-powered paraphrasing tool using T5 Transformer and LanguageTool
    for grammar, clarity, and fluency verification.
    """
    def __init__(self, model_name: str = "Vamsi/T5_Paraphrase_Paws"):
        print("Loading Transformer model and tokenizer (this may take a minute on first run)...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        # Enable GPU if available in Colab
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        print(f"Model loaded successfully on device: {self.device}")

        print("Initializing Grammar and NLP Checkers...")
        self.grammar_tool = language_tool_python.LanguageTool('en-US')
        self.nlp = spacy.load("en_core_web_sm")

    def generate_paraphrase(
        self,
        text: str,
        num_return_sequences: int = 1,
        temperature: float = 1.2,
        max_length: int = 256
    ) -> list:
        """Generates paraphrased text using T5."""
        text_formatted = f"paraphrase: {text} </s>"

        inputs = self.tokenizer(
            text_formatted,
            padding="longest",
            max_length=max_length,
            truncation=True,
            return_tensors="pt"
        ).to(self.device)

        outputs = self.model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=temperature,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

        return [self.tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    def verify_and_clean_text(self, text: str) -> dict:
        """Checks grammar, spelling, and readability."""
        matches = self.grammar_tool.check(text)
        corrected_text = language_tool_python.utils.correct(text, matches)
        doc = self.nlp(corrected_text)

        return {
            "original_paraphrase": text,
            "corrected_text": corrected_text,
            "grammar_issue_count": len(matches),
            "grammar_issues": [m.message for m in matches],
            "word_count": len([token for token in doc if not token.is_punct])
        }

    def paraphrase(self, text: str) -> dict:
        """Full pipeline execution."""
        raw_outputs = self.generate_paraphrase(text, num_return_sequences=1)
        return self.verify_and_clean_text(raw_outputs[0])

# Instantiate the object
paraphraser = AIParaphraser()


Loading Transformer model and tokenizer (this may take a minute on first run)...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Model loaded successfully on device: cpu
Initializing Grammar and NLP Checkers...


# **Build Evaluation Model**

In [11]:
# Step 3: Define the Evaluation Module
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

class ParaphraseEvaluator:
    """
    Evaluates paraphrased text against original text using:
    - ROUGE-1/2/L (Measures lexical variation and text overlap)
    - Semantic Similarity (Measures meaning preservation using Sentence-BERT)
    """
    def __init__(self):
        print("Loading evaluation model (Sentence-Transformers)...")
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        self.similarity_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        print("Evaluation module initialized successfully!")

    def evaluate(self, original_text: str, paraphrased_text: str) -> dict:
        # 1. ROUGE Scores
        rouge_scores = self.rouge.score(original_text, paraphrased_text)

        # Extract F1 score safely regardless of tuple or Score object format
        r1 = getattr(rouge_scores['rouge1'], 'f1', getattr(rouge_scores['rouge1'], 'mid', None).f1 if hasattr(rouge_scores['rouge1'], 'mid') else rouge_scores['rouge1'][2])
        r2 = getattr(rouge_scores['rouge2'], 'f1', getattr(rouge_scores['rouge2'], 'mid', None).f1 if hasattr(rouge_scores['rouge2'], 'mid') else rouge_scores['rouge2'][2])
        rl = getattr(rouge_scores['rougeL'], 'f1', getattr(rouge_scores['rougeL'], 'mid', None).f1 if hasattr(rouge_scores['rougeL'], 'mid') else rouge_scores['rougeL'][2])

        # 2. Semantic Similarity Score via Dense Vector Embeddings
        emb1 = self.similarity_model.encode(original_text, convert_to_tensor=True)
        emb2 = self.similarity_model.encode(paraphrased_text, convert_to_tensor=True)
        sim_score = util.cos_sim(emb1, emb2).item()

        return {
            "semantic_similarity": round(sim_score, 4),
            "rouge1_f1": round(rouge_scores['rouge1'].fmeasure, 4),
            "rouge2_f1": round(rouge_scores['rouge2'].fmeasure, 4),
            "rougeL_f1": round(rouge_scores['rougeL'].fmeasure, 4),
            "originality_score": round(1.0 - rouge_scores['rouge1'].fmeasure, 4)
        }

# Instantiate the evaluator object
evaluator = ParaphraseEvaluator()

Loading evaluation model (Sentence-Transformers)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluation module initialized successfully!


# **Run Sample Demonstrations & View Results**

In [12]:
# Step 4: Run Sample Test Cases & Display Evaluation
samples = [
    "Artificial intelligence is transforming how modern industries operate by automating complex tasks.",
    "To achieve long-term success in software engineering, developers must commit to continuous learning and code quality.",
    "Climate change poses significant risks to global agriculture, threatening food security in vulnerable regions."
]

print("=" * 80)
print("              PARAPHRASING & EVALUATION RESULTS REPORT")
print("=" * 80 + "\n")

for idx, sample in enumerate(samples, 1):
    print(f"[{idx}] ORIGINAL TEXT:")
    print(f"    \"{sample}\"\n")

    # Run pipeline
    result = paraphraser.paraphrase(sample)
    eval_metrics = evaluator.evaluate(sample, result['corrected_text'])

    print(f"    PARAPHRASED OUTPUT:")
    print(f"    \"{result['corrected_text']}\"\n")

    print("    --- Quality & Grammar Verification ---")
    print(f"    Grammar Issues Detected & Fixed : {result['grammar_issue_count']}")
    print(f"    Word Count                      : {result['word_count']}\n")

    print("    --- Evaluation Metrics ---")
    print(f"    Semantic Similarity (Meaning)   : {eval_metrics['semantic_similarity']} / 1.0000")
    print(f"    Originality Score (Diversity)   : {eval_metrics['originality_score']}")
    print(f"    ROUGE-1 F1 Score                : {eval_metrics['rouge1_f1']}")
    print(f"    ROUGE-L F1 Score                : {eval_metrics['rougeL_f1']}")
    print("-" * 80 + "\n")

              PARAPHRASING & EVALUATION RESULTS REPORT

[1] ORIGINAL TEXT:
    "Artificial intelligence is transforming how modern industries operate by automating complex tasks."

    PARAPHRASED OUTPUT:
    "By automating complex tasks, the artificial intelligence has transformed how modern industries operate."

    --- Quality & Grammar Verification ---
    Grammar Issues Detected & Fixed : 1
    Word Count                      : 13

    --- Evaluation Metrics ---
    Semantic Similarity (Meaning)   : 0.9588 / 1.0000
    Originality Score (Diversity)   : 0.12
    ROUGE-1 F1 Score                : 0.88
    ROUGE-L F1 Score                : 0.56
--------------------------------------------------------------------------------

[2] ORIGINAL TEXT:
    "To achieve long-term success in software engineering, developers must commit to continuous learning and code quality."

    PARAPHRASED OUTPUT:
    "To achieve long-term success in software engineering, developers must commit to continuous

In [13]:
# Interactive Text Paraphraser Cell
user_text = input("Enter any sentence to paraphrase: ")

if user_text.strip():
    result = paraphraser.paraphrase(user_text)
    metrics = evaluator.evaluate(user_text, result['corrected_text'])

    print("\n" + "="*50)
    print("OUTPUT RESULTS")
    print("="*50)
    print(f"Original:    {user_text}")
    print(f"Paraphrase:  {result['corrected_text']}")
    print(f"Similarity:  {metrics['semantic_similarity']} / 1.0000")
    print(f"Originality: {metrics['originality_score']}")
    print("="*50)

Enter any sentence to paraphrase: Planning meals in advance and making a grocery list helps save both time and money during weekly shopping trips.

OUTPUT RESULTS
Original:    Planning meals in advance and making a grocery list helps save both time and money during weekly shopping trips.
Paraphrase:  Planning meals in advance and writing a grocery list helps to save time as well as money during weekly shopping trips.
Similarity:  0.9787 / 1.0000
Originality: 0.2


# Project Evaluation Report: AI Paraphrasing Engine
 Model Architecture & Overview
Paraphrasing Model: Pre-trained T5-base sequence-to-sequence transformer fine-tuned on sentence rephrasing (Vamsi/T5_Paraphrase_Paws).

Post-Processing & Quality Assurance: Integrated LanguageTool engine and spaCy NLP pipeline to check grammar, spelling, and sentence fluency on generated sequences.

Semantic Evaluation Engine: Sentence-Transformers (all-MiniLM-L6-v2) dense vector embeddings to calculate cosine similarity between source and target text.


# Output Analysis & Benchmark Results

Sample 1: The AI changed the sentence order (putting "By automating complex tasks..." at the start) without changing the meaning. It achieved a 0.9588 similarity score and fixed 1 grammar issue.

Sample 2: The AI changed how a phrase was written (switching "code quality" to "the quality of code"). It scored a very high 0.9892 similarity score and fixed 1 grammar issue.

Sample 3: The AI replaced words with synonyms (changing "poses risks" to "presents threats"). It reached an originality score of 0.2000 (showing more unique wording) with a 0.9595 similarity score and fixed 1 grammar issue.


# Key Findings & Metric Interpretation

Meaning Preservation: High average cosine similarity (0.9691) proves that the dense embedding space of Sentence-BERT accurately tracks semantic fidelity without context loss.Lexical & Structural Variation: Lower ROUGE-1 scores correlate with higher structural rewording (e.g., Sample 3 achieved an Originality Score of 0.2000 through active verb substitutions).Automated Quality Control: LanguageTool identified and smoothed out surface-level fluency issues across all generated candidates before producing the final string output.